In [23]:
from tokenizers import BertWordPieceTokenizer
from pathlib import Path
vocab_size = 30000
corpus_path = Path("/home/eros/Documents/voynich-nlp-analysis/data/full_text.txt")
tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=False,
    lowercase=True,
)

tokenizer.train(
    files=str(corpus_path),
    vocab_size=vocab_size,
    min_frequency=2,
    limit_alphabet=1000,
    wordpieces_prefix="##"
)

tokenizer.save_model("/home/eros/Documents/voynich-nlp-analysis/data/models")

['/home/eros/Documents/voynich-nlp-analysis/data/models/vocab.txt']

In [24]:
from transformers import BertConfig, BertForMaskedLM
config = BertConfig(
    vocab_size=vocab_size,
    max_position_embeddings=512,
    hidden_size=256,
    num_attention_heads=4,
    num_hidden_layers=2,
    type_vocab_size=2,
)
model = BertForMaskedLM(config)

In [26]:
from datasets import Dataset
from transformers import BertTokenizerFast
def load_dataset():
    dataset_path = "/home/eros/Documents/voynich-nlp-analysis/data/full_text.txt"
    with open(dataset_path) as f:
        data = f.read().splitlines()
        lines = [line.strip() for line in data if line.strip()]
    return lines

def create_examples(lines):
    return [{"text": line} for line in lines if len(line.split()) > 10]

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

lines = load_dataset()
examples = create_examples(lines)
dataset = Dataset.from_list(examples)
tokenizer = BertTokenizerFast.from_pretrained("/home/eros/Documents/voynich-nlp-analysis/data/models")
tokenized_dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 1107/1107 [00:00<00:00, 7104.83 examples/s]


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="/home/eros/Documents/voynich-nlp-analysis/data/models/bert-custom",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=100,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()
